# Interactive TXC Integrator Explorer

This notebook lets you interactively change two parameter lists and regenerate four plots in a 2x2 layout.

Clockwise from top-left, the plots are:
1. $q$ vs axial strain
2. $q$ vs $p'$
3. $e$ vs $\log_{10}(p')$
4. Volumetric strain vs axial strain

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

workspace_root = Path.cwd()
package_src = workspace_root / "m02_tx_integrator" / "src"
if str(package_src) not in sys.path:
    sys.path.insert(0, str(package_src))

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as exc:
    raise ImportError(
        "ipywidgets is required for interactivity. Install it with `%pip install ipywidgets`."
    ) from exc

from m02_tx_integrator import TXCIntegrator, TestConditions
from m02_tx_integrator.parameters import MaterialParameters, SmallStrainParameters
from m02_tx_integrator.presets import nevada_sand_original

## Parameter Controls

Edit these parameter lists and click **Run / Update Plots**.

Primary controls (default workflow):
- `material_parameters`: `[ecsref, l, csi, pref (kPa), mcs, k1, l1]`
- `initial_conditions`: `[e0, p0 (kPa), q0 (kPa)]`

Optional advanced controls:
- `stiffness_parameters`: `[g0 (kPa), mg, a0 (m/m), b, rgmin, Poisson's ratio]`

Fixed model settings in this notebook:
- `k2 = 0.0`
- `l2 = 0.0`

In [ ]:
default_material, default_stiffness = nevada_sand_original()

material_names = ["ecsref", "l", "csi", "pref", "mcs", "k1", "l1"]
material_defaults = [
    default_material.ecsref,
    default_material.l,
    default_material.csi,
    default_material.pref,
    default_material.mcs,
    default_material.k1,
    default_material.l1,
]

initial_names = ["e0", "p0", "q0"]
initial_defaults = [0.728, 40.0, 0.0]

stiffness_names = ["g0", "mg", "a0", "b", "rgmin", "pr"]
stiffness_defaults = [
    default_stiffness.g0,
    default_stiffness.mg,
    default_stiffness.a0,
    default_stiffness.b,
    default_stiffness.rgmin,
    default_stiffness.pr,
]

material_labels = {
    "ecsref": "ecsref",
    "l": "l",
    "csi": "csi",
    "pref": "pref (kPa)",
    "mcs": "mcs",
    "k1": "k1",
    "l1": "l1",
}
initial_labels = {
    "e0": "e0",
    "p0": "p0 (kPa)",
    "q0": "q0 (kPa)",
}
stiffness_labels = {
    "g0": "g0 (kPa)",
    "mg": "mg",
    "a0": "a0 (m/m)",
    "b": "b",
    "rgmin": "rgmin",
    "pr": "Poisson's ratio",
}

material_widgets = {
    name: widgets.FloatText(
        value=value,
        description=material_labels[name],
        layout=widgets.Layout(width="300px"),
        style={"description_width": "130px"},
    )
    for name, value in zip(material_names, material_defaults)
}
initial_widgets = {
    name: widgets.FloatText(
        value=value,
        description=initial_labels[name],
        layout=widgets.Layout(width="300px"),
        style={"description_width": "130px"},
    )
    for name, value in zip(initial_names, initial_defaults)
}
stiffness_widgets = {
    name: widgets.FloatText(
        value=value,
        description=stiffness_labels[name],
        layout=widgets.Layout(width="300px"),
        style={"description_width": "130px"},
    )
    for name, value in zip(stiffness_names, stiffness_defaults)
}

label_widget = widgets.Text(value="Interactive-Case", description="label", layout=widgets.Layout(width="360px"))
run_button = widgets.Button(description="Run / Update Plots", button_style="success")
reset_button = widgets.Button(description="Reset Defaults", button_style="")
output = widgets.Output()

def _current_material_list():
    return [material_widgets[name].value for name in material_names]

def _current_initial_list():
    return [initial_widgets[name].value for name in initial_names]

def _current_stiffness_list():
    return [stiffness_widgets[name].value for name in stiffness_names]

def _build_material_from_widgets():
    values = _current_material_list()
    return MaterialParameters(
        ecsref=values[0],
        l=values[1],
        csi=values[2],
        pref=values[3],
        mcs=values[4],
        k1=values[5],
        k2=0.0,
        l1=values[6],
        l2=0.0,
    )

def _build_stiffness_from_widgets():
    values = _current_stiffness_list()
    return SmallStrainParameters(
        g0=values[0],
        mg=values[1],
        a0=values[2],
        b=values[3],
        rgmin=values[4],
        pr=values[5],
    )

def _build_test_from_widgets():
    values = _current_initial_list()
    return TestConditions(label=label_widget.value, e0=values[0], p0=values[1], q0=values[2])

def _render_plots(_=None):
    with output:
        clear_output(wait=True)
        try:
            material = _build_material_from_widgets()
            stiffness = _build_stiffness_from_widgets()
            test = _build_test_from_widgets()
            integrator = TXCIntegrator(material, stiffness)
            result = integrator.run(test)

            material_parameters = _current_material_list()
            initial_conditions = _current_initial_list()
            stiffness_parameters = _current_stiffness_list()
            print("material_parameters  =", material_parameters)
            print("initial_conditions   =", initial_conditions)
            print("stiffness_parameters =", stiffness_parameters)
            print("fixed parameters     = {'k2': 0.0, 'l2': 0.0}")

            fig, axes = plt.subplots(3, 2, figsize=(11, 13))

            ax_tl = axes[0, 0]
            ax_tr = axes[0, 1]
            ax_ml = axes[1, 0]
            ax_mr = axes[1, 1]
            ax_bl = axes[2, 0]
            ax_br = axes[2, 1]

            p_arr = np.asarray(result.p, dtype=float)
            q_arr = np.asarray(result.q, dtype=float)
            ed_arr = np.asarray(result.ed, dtype=float)
            evol_arr = np.asarray(result.evol, dtype=float) / 100.0
            pvr_arr = np.asarray(result.pvr, dtype=float)
            gtan_arr = np.asarray(result.gtan, dtype=float)
            p_safe = np.maximum(p_arr, 1e-9)
            stress_ratio = q_arr / p_safe

            e0_ref = pvr_arr[0]
            eps_v_p = -(pvr_arr - e0_ref) / (1.0 + e0_ref)

            dq = np.diff(q_arr)
            g_prev = np.maximum(gtan_arr[:-1], 1e-12)
            d_eps_d_e = dq / (np.sqrt(3.0) * g_prev)
            eps_d_e = np.concatenate(([0.0], np.cumsum(d_eps_d_e)))
            eps_d_p = ed_arr - eps_d_e

            d_eps_v_p = np.diff(eps_v_p)
            d_eps_d_p = np.diff(eps_d_p)
            with np.errstate(divide='ignore', invalid='ignore'):
                dil_rate_plastic = d_eps_v_p / d_eps_d_p
            dil_rate_plastic[np.abs(d_eps_d_p) < 1e-12] = np.nan

            d_eps_v_total = np.diff(evol_arr)
            d_eps_d_total = np.diff(ed_arr)
            with np.errstate(divide='ignore', invalid='ignore'):
                dil_rate_total = d_eps_v_total / d_eps_d_total
            dil_rate_total[np.abs(d_eps_d_total) < 1e-12] = np.nan

            stress_ratio_mid = stress_ratio[1:]

            ax_tl.plot(result.eax, result.q, color="#005f73", lw=2.0)
            ax_tl.set_title("q - axial strain")
            ax_tl.set_xlabel("axial strain (%)")
            ax_tl.set_ylabel("q (kPa)")
            ax_tl.set_xlim(0.0, 30.0)
            ax_tl.grid(alpha=0.25)

            qp_min = min(float(np.min(p_arr)), float(np.min(q_arr)))
            qp_max = max(float(np.max(p_arr)), float(np.max(q_arr)))
            qp_span = max(qp_max - qp_min, 1e-9)
            qp_pad = 0.05 * qp_span
            qp_lo = max(0.0, qp_min - qp_pad)
            qp_hi = qp_max + qp_pad

            ax_tr.plot(result.p, result.q, color="#0a9396", lw=2.0)
            ax_tr.set_title("q - p'")
            ax_tr.set_xlabel("p' (kPa)")
            ax_tr.set_ylabel("q (kPa)")
            ax_tr.set_xlim(qp_lo, qp_hi)
            ax_tr.set_ylim(qp_lo, qp_hi)
            ax_tr.set_aspect("equal", adjustable="box")
            ax_tr.grid(alpha=0.25)

            p_lo = float(10 ** np.floor(np.log10(np.min(p_safe))))
            p_hi = float(10 ** np.ceil(np.log10(np.max(p_safe))))
            if p_hi <= p_lo:
                p_hi = p_lo * 10.0
            p_csl = np.logspace(np.log10(p_lo), np.log10(p_hi), 200)
            e_csl = material.ecsref - material.l * (p_csl / material.pref) ** material.csi

            ax_mr.plot(p_safe, result.vr, color="#94d2bd", lw=2.0, label="Response")
            ax_mr.plot(p_csl, e_csl, color="#bb3e03", lw=2.0, ls="--", label="Critical state line")
            ax_mr.set_title("e - p'")
            ax_mr.set_xscale("log")
            ax_mr.set_xlim(p_lo, p_hi)
            ax_mr.set_xlabel("p' (kPa)")
            ax_mr.set_ylabel("e")
            ax_mr.grid(alpha=0.25, which="both")
            ax_mr.legend()

            ax_ml.plot(result.eax, result.evol, color="#ee9b00", lw=2.0)
            ax_ml.set_title("volumetric strain - axial strain")
            ax_ml.set_xlabel("axial strain (%)")
            ax_ml.set_ylabel("volumetric strain (%)")
            ax_ml.set_xlim(0.0, 30.0)
            ax_ml.grid(alpha=0.25)

            ax_bl.plot(result.eax, stress_ratio, color="#3a86ff", lw=2.0)
            ax_bl.set_title("stress ratio (q/p') - axial strain")
            ax_bl.set_xlabel("axial strain (%)")
            ax_bl.set_ylabel("q/p'")
            ax_bl.set_xlim(0.0, 30.0)
            ax_bl.grid(alpha=0.25)

            ax_br.plot(
                dil_rate_plastic,
                stress_ratio_mid,
                color="#8338ec",
                lw=1.8,
                label="Plastic: dεv_p / dεd_p",
            )
            ax_br.plot(
                dil_rate_total,
                stress_ratio_mid,
                color="#fb5607",
                lw=1.8,
                ls="--",
                label="Total: dεv / dεd",
            )
            ax_br.set_title("stress ratio (q/p') - dilatancy rate")
            ax_br.set_xlabel("dilatancy rate")
            ax_br.set_ylabel("q/p'")
            ax_br.grid(alpha=0.25)
            ax_br.legend()

            fig.suptitle(f"TXC response: {result.label}", fontsize=13, y=0.99)
            fig.tight_layout()
            plt.show()

        except Exception as exc:
            print("Plot update failed:", type(exc).__name__, str(exc))

def _reset_defaults(_=None):
    for name, value in zip(material_names, material_defaults):
        material_widgets[name].value = value
    for name, value in zip(initial_names, initial_defaults):
        initial_widgets[name].value = value
    for name, value in zip(stiffness_names, stiffness_defaults):
        stiffness_widgets[name].value = value
    label_widget.value = "Interactive-Case"
    _render_plots()

run_button.on_click(_render_plots)
reset_button.on_click(_reset_defaults)

controls_material = widgets.VBox(
    [widgets.HTML("<b>material_parameters</b>")] + [material_widgets[name] for name in material_names]
 )
controls_initial = widgets.VBox(
    [widgets.HTML("<b>initial_conditions</b>"), label_widget] + [initial_widgets[name] for name in initial_names]
 )
controls_stiffness = widgets.VBox(
    [widgets.HTML("<b>stiffness_parameters (optional)</b>")] + [stiffness_widgets[name] for name in stiffness_names]
 )

display(widgets.HBox([controls_material, controls_initial, controls_stiffness]))
display(widgets.HBox([run_button, reset_button]))
display(output)

_render_plots()

Output()